# Q7 - Intel Image Classification using Transfer Learning (PyTorch)
This notebook demonstrates image classification on the Intel Image Classification dataset using a pretrained ResNet18 model.

## Install and Download Dataset

In [ ]:
!pip install -q kaggle
# Upload kaggle.json and then run:
# !mkdir -p ~/.kaggle
# !cp kaggle.json ~/.kaggle/
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d puneet6060/intel-image-classification
# !unzip intel-image-classification.zip

## Imports

In [ ]:
import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from PIL import Image

## Data Preprocessing and Augmentation

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

In [ ]:
train_dataset = datasets.ImageFolder('seg_train/seg_train', transform=train_transform)
test_dataset = datasets.ImageFolder('seg_test/seg_test', transform=test_transform)

class_names = train_dataset.classes
print(class_names)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## Transfer Learning using ResNet18

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = models.resnet18(pretrained=True)

for param in model.parameters():
    param.requires_grad = False

num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 6)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

In [ ]:
epochs = 10
train_loss, train_acc = [], []

for epoch in range(epochs):
    model.train()
    running_loss, correct, total = 0, 0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, pred = torch.max(outputs,1)
        total += labels.size(0)
        correct += (pred == labels).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    train_loss.append(epoch_loss)
    train_acc.append(epoch_acc)

    print(f'Epoch {epoch+1}: Loss={epoch_loss:.4f}, Accuracy={epoch_acc:.2f}%')

## Evaluation

In [ ]:
model.eval()
correct, total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, pred = torch.max(outputs,1)

        total += labels.size(0)
        correct += (pred == labels).sum().item()

        all_preds.extend(pred.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

print('Test Accuracy:', 100 * correct / total)

In [ ]:
plt.plot(train_acc)
plt.title('Training Accuracy')
plt.show()

plt.plot(train_loss)
plt.title('Training Loss')
plt.show()

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=class_names,
            yticklabels=class_names)
plt.show()

print(classification_report(all_labels, all_preds,
                            target_names=class_names))

## Save and Load Model

In [ ]:
def save_model(model, path):
    torch.save(model.state_dict(), path)

def load_model(path):
    model = models.resnet18(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 6)
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

save_model(model, 'Q7_trained_model.pth')

In [ ]:
def predict_image(image_path):
    image = Image.open(image_path).convert('RGB')
    image = test_transform(image).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        output = model(image)
        _, pred = torch.max(output,1)

    return class_names[pred.item()]